# Phase 0: τ-bench setup and baseline pilot

**Goal:** get τ-bench running for free, and check if the baseline is strong enough to use.

**Setup**
- Agent: `qwen/qwen3.8-27b` on Groq (free tier, preview model, picked 2026-10-24). `llama-3.3-70b-versatile` was the plan, but my key can't use it. Qwen is the strongest model left on my key's list.
- User simulator: `openai/gpt-oss-120b` on Groq (free tier)
- Domain: airline, tasks 0–2, one try each (k=1)
- System: stock agent only (the baseline)

The user model stays fixed for all 5 systems. τ-bench does the same: it fixes the user model no matter which agent it tests.

**How to run:** run every cell from top to bottom. After a restart, run them all again.

In [ ]:
# mount Google Drive so my results are saved
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# clone tau-bench only if the folder is not there yet
import os
REPO = "/content/drive/MyDrive/cmsc691-10-project2"
if not os.path.exists(REPO):
    !git clone https://github.com/sierra-research/tau-bench.git {REPO}
else:
    print("repo already here")

repo already here


In [ ]:
# go into the repo, where run.py is
# every later cell runs from here
%cd /content/drive/MyDrive/cmsc691-10-project2

# Colab drops packages on restart, so I run this every time
!pip install -q -e .
# new litellm needs the mcp package, tau-bench does not list it
!pip install -q mcp
print("installed")

/content/drive/MyDrive/cmsc691-10-project2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for tau_bench (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-monitoring 2.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-dataproc 5.32.0 requires pro

In [ ]:
# get the Groq key from Colab Secrets
# never paste the key into a file
import os
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
print('key loaded')

key loaded


In [ ]:
# list the models my Groq key can use
# the agent model must be in this list
import os, requests
r = requests.get("https://api.groq.com/openai/v1/models",
                 headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"})
for m in sorted(x["id"] for x in r.json()["data"]):
    print(m)

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
whisper-large-v3
whisper-large-v3-turbo


In [ ]:
# pick the agent model from the list above
# it must be different from the user model (gpt-oss-120b)
AGENT_MODEL = "qwen/qwen3.8-27b"
print("agent:", AGENT_MODEL)

agent: qwen/qwen3.8-27b


In [ ]:
%%writefile /content/drive/MyDrive/cmsc691-10-project2/pilot_run.py
# tau-bench gives reward 0 for ANY error, rate limits too
# that makes the baseline look worse than it is
# Groq free tier only allows ~7000 input tokens a minute,
# and one call is ~4000+, so I wait the time Groq asks for, then retry
import re, sys, time, litellm

# tau-bench puts the user model name in the result file name
# a "/" in the name breaks the path, so I use a short name
litellm.model_alias_map = {"gpt-oss-120b": "openai/gpt-oss-120b"}

_real = litellm.completion

def patient_completion(*args, **kwargs):
    for attempt in range(30):
        try:
            return _real(*args, **kwargs)
        except litellm.RateLimitError as e:
            msg = str(e)
            # one call bigger than the per-minute limit: waiting won't help
            m = re.search(r"Limit (\d+), Used \d+, Requested (\d+)", msg)
            if m and int(m.group(2)) > int(m.group(1)):
                raise
            # wait the time Groq asks for, plus a little
            w = re.search(r"try again in ([\d.]+)s", msg)
            wait = float(w.group(1)) + 2 if w else 30
            print(f"  rate limit, waiting {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError("still rate limited after 30 tries")

# swap it in BEFORE tau-bench imports completion
litellm.completion = patient_completion

sys.argv[0] = "run.py"
from run import main
main()

Overwriting /content/drive/MyDrive/cmsc691-10-project2/pilot_run.py


In [ ]:
%%writefile /content/drive/MyDrive/cmsc691-10-project2/summarize.py
# count API errors on their own
# they are not real task failures, so I leave them out of pass^1
import json, sys

rows = json.load(open(sys.argv[1]))
err = [r for r in rows if "error" in (r.get("info") or {})]
ok = [r for r in rows if r not in err]
p = sum(r["reward"] == 1 for r in ok)

print(f"total {len(rows)} | API errors {len(err)} | valid {len(ok)} | passed {p}")
if ok:
    print(f"pass^1 = {p/len(ok):.2f}")

# 429 = too many calls this minute, retry helps
# 413 = one call too big, retry won't help
for r in err:
    print("error task", r["task_id"], r["info"]["error"][:120])

Overwriting /content/drive/MyDrive/cmsc691-10-project2/summarize.py


In [ ]:
# baseline pilot: airline, tasks 0-2, one try each
# agent: AGENT_MODEL from the cell above
# user: gpt-oss-120b, fixed for all 5 systems
# concurrency 1 keeps me under the free limits
%cd /content/drive/MyDrive/cmsc691-10-project2
!python pilot_run.py --env airline --agent-strategy tool-calling \
  --model {AGENT_MODEL} --model-provider groq \
  --user-model gpt-oss-120b --user-model-provider groq \
  --user-strategy llm --num-trials 1 \
  --temperature 0.0 --task-ids 0 1 2 \
  --max-concurrency 1 --log-dir results_pilot

/content/drive/MyDrive/cmsc691-10-project2
Namespace(num_trials=1, env='airline', model='qwen/qwen3.8-27b', model_provider='groq', user_model='gpt-oss-120b', user_model_provider='groq', agent_strategy='tool-calling', temperature=0.0, task_split='test', start_index=0, end_index=-1, task_ids=[0, 1, 2], log_dir='results_pilot', max_concurrency=1, seed=10, shuffle=0, user_strategy='llm', few_shot_displays_path=None)
Loading user with strategy: llm
Running tasks [0, 1, 2] (checkpoint path: results_pilot/tool-calling-qwen3.8-27b-0.0_range_0--1_user-gpt-oss-120b-llm_1024182814.json)
Running task 0

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.

  rate limit, waiting 12s

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.

  rate limit, waiting 38s

Give Feedback / Get Help: https://github.com/

In [15]:
# read the newest result file
%cd /content/drive/MyDrive/cmsc691-10-project2
!python summarize.py $(ls -t results_pilot/*.json | head -1)

/content/drive/MyDrive/cmsc691-10-project2
total 2 | API errors 1 | valid 1 | passed 0
pass^1 = 0.00
error task 0 still rate limited after 30 tries


## Summary

**What I did**
- Cloned τ-bench into Drive and installed it.
- Loaded the Groq key from Colab Secrets. No key in any file.
- Added `pilot_run.py`. Without it, τ-bench scores a rate-limit error as a failed task.
- Added `summarize.py`. It counts API errors apart from real failures.
- Found my key can't use `llama-3.3-70b-versatile`. I listed the models my key can use and picked `qwen/qwen3.8-27b` (preview, 2026-10-24).
- Fixed a crash: τ-bench puts the user model name in the file name, and the `/` in `openai/gpt-oss-120b` broke the path. I gave it a short name.
- Added the missing `mcp` package. New litellm needs it.
- First qwen run hit Groq's limit at once: 7000 input tokens a minute, and one call was 4135. litellm's own retry gave up in seconds. So `pilot_run.py` now waits the time Groq asks for, then retries.
- Ran the baseline on airline tasks 0–2 on 2026-10-24.

**Why free Groq, and its limits**
- GPT-4o for the full study costs real money. I want $0.
- Groq free tier is tight. qwen3.8-27b allows only 7000 input tokens a minute. One call starts at ~4100 tokens and grows as the dialogue grows.
- The full study is 5 systems × 50 tasks × 8 tries = 2000 runs.

**Results**
- I ran for about 1 hour. Only 1 of 3 tasks finished.
- Task 0: still rate limited after 30 tries. Counted as an API error.
- Task 1: finished, reward 0. A real failure, not an API error.
- Task 2: I stopped it by hand. It was still waiting on the limit.
- pass^1 on valid runs: 0/1. One task is too few to say anything about the baseline.

**Analysis**
- Most waits were the default 30s. My code only reads waits like "8.6s". Groq likely asked for minutes ("12m30s"), so my code missed it. Minute-long waits point to a daily limit, not just the per-minute one.
- I can't confirm this. My wrapper replaced Groq's error text with its own message, so the original was lost.
- Either way, the speed is the problem. About 1 task an hour means 2000 runs would take months.
- I saw no call over 7000 tokens, but the dialogues never got long enough to test it.

**Decision**
- Groq free tier can't run the main study. I stop using it here.
- Next I run the agent myself with vLLM on Colab Pro. No rate limits, still $0.

**Lessons for next time**
- Keep the API's original error text in the log.
- Read wait times in minutes too, not only seconds.
- Check a provider's daily limit before the run, not after.

**Next steps**
1. Build a vLLM notebook on Colab Pro. Run Qwen3-8B on airline tasks 0–9, k=1.
2. Check the baseline. Near 0 means floor effect, and I need a stronger model.
3. Pick one agent model for all 5 systems.
4. Decide if the user simulator also moves to vLLM. It must stay fixed for all 5 systems.
5. Start Milestone 1: build the airline policy graph by hand from `wiki.md`.